# 01. Runtime 请求路径：HTTP 到 Scheduler 再回到用户

> 面向即将加入 SGLang 开源社区的开发者。建议从仓库根目录启动 Jupyter，
> 或者在 notebook 第一格运行路径检查。本文以本地源码为主，线上文档为索引。
> Markdown 中的源码摘录会插入 `# 注：...` 行内讲解；可执行代码 cell 则保持可运行。


In [ ]:
from pathlib import Path
import ast, inspect, re, textwrap

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "python" / "sglang").exists() and (candidate / "docs").exists():
            return candidate
    raise RuntimeError("没有找到 SGLang 仓库根目录，请从仓库内启动 notebook。")

ROOT = find_repo_root()
print(ROOT)

def read_rel(path):
    return (ROOT / path).read_text()

def show_lines(path, start, end):
    lines = read_rel(path).splitlines()
    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:4d}: {lines[i-1]}")

def find_defs(path, names=None):
    tree = ast.parse(read_rel(path))
    rows = []
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            if names is None or node.name in names:
                rows.append((node.lineno, type(node).__name__, node.name))
    return sorted(rows)


## 入口层：`sglang serve` / `python -m sglang.launch_server`

启动入口会解析 `ServerArgs`，然后根据模式分流到 HTTP、gRPC、Ray、encoder-only 或 disaggregation server。默认 HTTP 模式进入 `sglang.srt.entrypoints.http_server.launch_server`。


In [ ]:
for path in ["python/sglang/launch_server.py", "python/sglang/srt/entrypoints/http_server.py"]:
    print("\n==", path)
    for row in find_defs(path, names={"run_server", "launch_server", "generate_request"}):
        print(row)


In [ ]:
show_lines("python/sglang/launch_server.py", 12, 45)


### 启动分流：`run_server` 只决定拓扑，不做推理

```python
# python/sglang/launch_server.py:12-43
  12: suppress_noisy_warnings()
      # 注：关键调用：`suppress_noisy_warnings` - 启动入口先压掉第三方库的已知噪声 warning，让真正的启动错误更容易被看到。
  13: 
  14: 
  15: def run_server(server_args):
  16:     """Run the server based on server_args.grpc_mode and server_args.encoder_only."""
  17:     if server_args.encoder_only:
      # 注：encoder-only 服务只处理 embedding/encoder 类请求；不会启动普通生成式 HTTP runtime。
  18:         # For encoder disaggregation
  19:         if server_args.grpc_mode:
      # 注：选择 gRPC 协议入口，绕过默认 FastAPI/uvicorn endpoint。
  20:             from sglang.srt.disaggregation.encode_grpc_server import (
  21:                 serve_grpc_encoder,
  22:             )
  23: 
  24:             asyncio.run(serve_grpc_encoder(server_args))
      # 注：关键调用：`serve_grpc_encoder` - encoder-only + gRPC 的专用服务路径，用于 embedding/encoder 类模型而不是普通生成式 HTTP server。
  25:         else:
  26:             from sglang.srt.disaggregation.encode_server import launch_server
  27: 
  28:             launch_server(server_args)
      # 注：关键调用：`launch_server` - `run_server` 的默认 HTTP/SRT 路径；真正的 manager 子进程装配在 `http_server.launch_server` 里完成。
  29:     elif server_args.grpc_mode:
  30:         # TODO: Once the native Rust gRPC server starts alongside HTTP in the
  31:         # default path below (controlled by SGLANG_ENABLE_GRPC / SGLANG_GRPC_PORT),
  32:         # remove this legacy SMG path and the grpc_mode flag.
  33:         from sglang.srt.entrypoints.grpc_server import serve_grpc
  34: 
  35:         asyncio.run(serve_grpc(server_args))
      # 注：关键调用：`serve_grpc` - 普通生成模型的 gRPC 服务路径；它绕开默认 FastAPI/uvicorn HTTP 入口。
  36:     elif server_args.use_ray:
      # 注：Ray serving 使用分布式 actor 拓扑，启动路径不再由本进程直接装配 SRT 子进程。
  37:         # Ray mode: HTTP mode with Ray backend.
  38:         try:
  39:             from sglang.srt.ray.http_server import launch_server
  40:         except ImportError:
  41:             raise ImportError(
  42:                 "Ray is required for --use-ray mode. "
  43:                 "Install it with: pip install 'sglang[ray]'"
```

**读这段时抓住：**

- `encoder_only`、`grpc_mode`、`use_ray` 是互斥的服务形态分支；默认路径才进入 HTTP server。
- 这里没有 tokenizer、scheduler、model runner 的细节，说明启动入口的职责是选择 runtime 拓扑。
- 以后排查“为什么我的参数没有进入默认 HTTP server”时，先看这段分流条件。


### 默认 HTTP 路径的核心：`http_server.launch_server`

`run_server` 只是把默认分支交给这里；真正把 HTTP server、TokenizerManager、Scheduler 子进程、
Detokenizer 子进程装配起来的是 `launch_server`。下面这段代码很短，但它是默认服务模式的总装入口。

```python
# python/sglang/srt/entrypoints/http_server.py:2455-2497
# 阶段 1：声明可替换的构造函数。测试、Ray/fork、私有部署可以替换这些函数。
def launch_server(
    server_args: ServerArgs,
    # 注：server_args 是整个服务的配置对象；后续 engine 子进程和 HTTP server 都共享这份配置。
    init_tokenizer_manager_func: Callable = init_tokenizer_manager,
    # 注：可替换 TokenizerManager 初始化函数，测试或私有 fork 可以在这里注入自定义 manager。
    run_scheduler_process_func: Callable = run_scheduler_process,
    # 注：可替换 Scheduler 子进程入口；调度器和模型加载都在这条路径里。
    run_detokenizer_process_func: Callable = run_detokenizer_process,
    # 注：可替换 Detokenizer 子进程入口；输出 token 到文本的转换由它负责。
    execute_warmup_func: Callable = _execute_server_warmup,
    # 注：warmup 在 HTTP server lifecycle 中执行，用于提前触发模型/图/缓存准备。
    launch_callback: Optional[Callable[[], None]] = None,
    # 注：服务完成启动后可调用的回调，常用于测试或嵌入式部署通知外部系统。
):
    """
    Launch SRT (SGLang Runtime) Server.

    The SRT server consists of an HTTP server and an SRT engine.
    ...
    """
    # 阶段 2：先启动 SRT engine 的内部组件。
    # TokenizerManager 在主进程；Scheduler/Detokenizer 通常在子进程。
    (
        tokenizer_manager,
        template_manager,
        port_args,
        scheduler_init_result,
        subprocess_watchdog,
    ) = Engine._launch_subprocesses(...)
    # 注：返回的 tokenizer_manager 留在主进程；scheduler_init_result 携带 scheduler 回传的能力信息。

    # 阶段 3：再把 FastAPI/uvicorn 绑定到这些组件上，开始接 HTTP 请求。
    _setup_and_run_http_server(
        server_args,
        # 注：HTTP endpoint 会通过 global state 间接调用 tokenizer_manager。
        tokenizer_manager,
        template_manager,
        port_args,
        scheduler_init_result.scheduler_infos,
        # 注：scheduler_infos 让 HTTP/tokenizer 侧知道 max input length、model info 等运行时事实。
        subprocess_watchdog,
        execute_warmup_func=execute_warmup_func,
        launch_callback=launch_callback,
    )
```

**读这段时抓住：**

- `launch_server` 是默认 HTTP 服务的“装配函数”，不是请求处理函数。
- 它先启动 engine 子系统，再启动 HTTP server；这保证 API 开始监听前 scheduler/model 已经在加载或就绪流程中。
- 函数参数都是可注入的 callable，这也是测试和私有 fork 可以替换启动行为的原因。
- `scheduler_init_result.scheduler_infos` 会被传给 HTTP 层，后续 API 层才能知道 max input length 等 scheduler 能力。


### `Engine._launch_subprocesses` 上半段：准备环境、端口和 Scheduler

```python
# python/sglang/srt/entrypoints/engine.py:751-844
 751:     def _launch_subprocesses(
 752:         cls,
 753:         server_args: ServerArgs,
 754:         init_tokenizer_manager_func: Callable,
 755:         run_scheduler_process_func: Callable,
 756:         run_detokenizer_process_func: Callable,
 757:         port_args: Optional[PortArgs] = None,
 758:     ) -> Tuple[
 759:         TokenizerManager,
 760:         TemplateManager,
 761:         PortArgs,
 762:         SchedulerInitResult,
 763:         Optional[SubprocessWatchdog],
 764:     ]:
 765:         """Launch the TokenizerManager in the main process, the Scheduler in a subprocess, and the DetokenizerManager in another subprocess.
 766: 
 767:         Returns:
 768:             Tuple of (tokenizer_manager, template_manager, port_args, scheduler_init_result, subprocess_watchdog).
 769:         """
 770:         # Configure global environment
 771:         configure_logger(server_args)
      # 注：关键调用：`configure_logger` - 按 server args 配置 SRT 进程日志；这一步发生在子进程启动前，保证 scheduler/detokenizer 日志格式一致。
 772:         _set_envs_and_config(server_args)
      # 注：关键调用：`_set_envs_and_config` - 把 server args 转成运行时环境变量和全局配置，后续 model runner、通信库和 cache 逻辑会读取这些开关。
 773: 
 774:         # Defensive: ensure plugins loaded (may already be loaded by
 775:         # Engine.__init__ or CLI entry).
 776:         load_plugins()
      # 注：关键调用：`load_plugins` - 加载插件，让插件有机会注册模型、参数 hook 或 speculative/grammar 扩展。
 777: 
 778:         server_args.check_server_args()
      # 注：关键调用：`server_args.check_server_args` - 在进程启动前做参数一致性校验，避免子进程启动后才失败。
 779:         _set_gc(server_args)
      # 注：关键调用：`_set_gc` - 按 server args 调整 Python GC 行为，避免长生命周期服务在请求热路径上频繁触发不可控 GC pause。
 780: 
 781:         # Allocate ports for inter-process communications
 782:         if port_args is None:
      # 注：外部没有传入 IPC 端口时，由 engine 自己为 tokenizer/scheduler/detokenizer 分配通信端点。
 783:             port_args = PortArgs.init_new(server_args)
      # 注：`port_args` 接收 `PortArgs.init_new` 的结果：分配 tokenizer/scheduler/detokenizer 之间 IPC 通信所需的端口或 socket 名称。
 784:         logger.info(f"{server_args=}")
 785: 
 786:         # Start the engine info bootstrap server if per-rank info is needed.
 787:         engine_info_bootstrap_server = None
 788:         if (
 789:             server_args.remote_instance_weight_loader_start_seed_via_transfer_engine
 790:             and server_args.node_rank == 0
 791:         ):
 792:             bootstrap_port = server_args.engine_info_bootstrap_port
 793:             if not is_port_available(bootstrap_port):
 794:                 raise RuntimeError(
 795:                     f"engine_info_bootstrap_port {bootstrap_port} is already in use. "
 796:                     f"When running multiple instances on the same node, each instance must use a "
 797:                     f"different --engine-info-bootstrap-port."
 798:                 )
 799:             engine_info_bootstrap_server = EngineInfoBootstrapServer(
 800:                 host=server_args.host, port=bootstrap_port
 801:             )
 802: 
 803:         if (
 804:             server_args.reasoning_parser == "auto"
 805:             or server_args.tool_call_parser == "auto"
 806:         ):
 807:             resolve_auto_parsers(server_args)
      # 注：关键调用：`resolve_auto_parsers` - 在 scheduler 启动前解析 `reasoning_parser/tool_call_parser=auto`，让 tokenizer/template 侧不用再猜 parser 类型。
 808: 
 809:         # Launch scheduler processes
 810:         scheduler_init_result, scheduler_procs = cls._launch_scheduler_processes(
      # 注：`scheduler_init_result, scheduler_procs` 接收 `cls._launch_scheduler_processes` 的结果：启动 scheduler/model worker 进程；模型加载、TP/DP 通信组和 scheduler capability 都从这里产生。
 811:             server_args, port_args, run_scheduler_process_func
 812:         )
 813:         scheduler_init_result.engine_info_bootstrap_server = (
 814:             engine_info_bootstrap_server
 815:         )
 816: 
 817:         if (
 818:             server_args.enable_elastic_expert_backup
 819:             and server_args.elastic_ep_backend is not None
 820:         ):
 821:             run_expert_backup_manager(server_args, port_args)
      # 注：关键调用：`run_expert_backup_manager` - 启动 DeepEP MoE expert 备份管理器，只在启用 expert distribution recorder 且当前节点需要维护冗余专家时运行。
 822: 
 823:         if server_args.node_rank >= 1:
      # 注：非 0 节点只承载 scheduler/model worker，不启动 tokenizer/detokenizer 和完整 HTTP API。
 824:             # In multi-node cases, non-zero rank nodes do not need to run tokenizer or detokenizer,
 825:             # so they can just wait here.
 826:             scheduler_init_result.wait_for_ready()
      # 注：关键调用：`scheduler_init_result.wait_for_ready` - 等待 scheduler/model worker 完成初始化并回传可服务状态。
 827: 
 828:             if os.getenv("SGLANG_BLOCK_NONZERO_RANK_CHILDREN") == "0":
 829:                 # When using `Engine` as a Python API, we don't want to block here.
 830:                 return (
 831:                     None,
 832:                     None,
 833:                     port_args,
 834:                     scheduler_init_result,
 835:                     None,
 836:                 )
 837: 
 838:             launch_dummy_health_check_server(
      # 注：关键调用：`launch_dummy_health_check_server` - 非 0 rank 不承载完整 HTTP API 时仍暴露健康检查端口，方便外部编排系统判断子节点存活。
 839:                 server_args.host, server_args.port, server_args.enable_metrics
 840:             )
 841: 
 842:             scheduler_init_result.wait_for_completion()
 843:             return (
 844:                 None,
```

**读这段时抓住：**

- `configure_logger`、`_set_envs_and_config`、`server_args.check_server_args()` 是进程启动前的全局准备。
- `PortArgs.init_new` 分配 manager 间 IPC 地址；后面的 Tokenizer/Scheduler/Detokenizer 都靠它通信。
- `_launch_scheduler_processes` 先启动 scheduler，因为模型加载和 scheduler 能力信息都来自那里。
- 多节点非 0 rank 可能只跑 scheduler/worker，不跑 tokenizer/detokenizer；这解释了为什么 HTTP API 通常只在 rank0 暴露。


### `Engine._launch_subprocesses` 下半段：Detokenizer、TokenizerManager、ready 信号和 watchdog

```python
# python/sglang/srt/entrypoints/engine.py:845-891
 845:                 None,
 846:                 port_args,
 847:                 scheduler_init_result,
 848:                 None,
 849:             )
 850: 
 851:         # Launch detokenizer process(es) — optionally fronted by a router when
 852:         # detokenizer_worker_num > 1.
 853:         detoken_procs, detoken_names = cls._launch_detokenizer_subprocesses(
      # 注：`detoken_procs, detoken_names` 接收 `cls._launch_detokenizer_subprocesses` 的结果：启动 detokenizer 进程，把 token id 到文本的 CPU 工作从 scheduler 中拆出去。
 854:             server_args=server_args,
 855:             port_args=port_args,
 856:             run_detokenizer_process_func=run_detokenizer_process_func,
 857:         )
 858:         for p in detoken_procs:
 859:             scheduler_init_result.all_child_pids.append(p.pid)
      # 注：关键调用：`scheduler_init_result.all_child_pids.append` - 把 detokenizer 子进程 PID 纳入 scheduler 初始化结果，后续 watchdog/清理逻辑才能统一管理。
 860: 
 861:         # Init tokenizer manager first, as the bootstrap server is initialized here
 862:         if server_args.tokenizer_worker_num == 1:
      # 注：单 tokenizer worker 模式下主进程直接持有 TokenizerManager；多 worker 模式改用共享内存和 router。
 863:             tokenizer_manager, template_manager = init_tokenizer_manager_func(
 864:                 server_args, port_args
 865:             )
 866:         else:
 867:             # Launch multi-tokenizer router
 868:             tokenizer_manager = MultiTokenizerRouter(server_args, port_args)
      # 注：主进程中的 TokenizerManager，负责 API 入站、tokenization 和等待 scheduler/detokenizer 回包。
 869:             template_manager = None
      # 注：chat/completion 模板管理器，初始化后还会给 auto parser 提供推断结果。
 870: 
 871:         # Wait for the model to finish loading
 872:         scheduler_init_result.wait_for_ready()
      # 注：关键调用：`scheduler_init_result.wait_for_ready` - 等待 scheduler/model worker 完成初始化并回传可服务状态。
 873: 
 874:         # Get back some info from scheduler to tokenizer_manager
 875:         tokenizer_manager.max_req_input_len = scheduler_init_result.scheduler_infos[0][
      # 注：scheduler 初始化后回传真实 max input length；tokenizer manager 后续用它做入站长度校验。
 876:             "max_req_input_len"
 877:         ]
 878: 
 879:         # Set up subprocess liveness watchdog to detect crashes
 880:         # Note: RayEngine returns scheduler_procs=None as it uses Ray actors instead of mp.Process
 881:         processes = list(scheduler_procs or [])
 882:         names = [f"scheduler_{i}" for i in range(len(processes))]
 883:         processes.extend(detoken_procs)
      # 注：关键调用：`processes.extend` - 把 detokenizer 进程追加到 watchdog 监控集合，主进程会同时观察 scheduler 与 detokenizer。
 884:         names.extend(detoken_names)
      # 注：关键调用：`names.extend` - 给 watchdog 中新增的 detokenizer 进程补上可读名称，崩溃日志能指出是哪类子进程失败。
 885:         subprocess_watchdog = SubprocessWatchdog(
      # 注：`subprocess_watchdog` 接收 `SubprocessWatchdog` 的结果：监控 scheduler/detokenizer 子进程存活，避免主进程静默挂着坏服务。
 886:             processes=processes, process_names=names
 887:         )
 888:         subprocess_watchdog.start()
      # 注：关键调用：`subprocess_watchdog.start` - 启动后台 watchdog 线程，子进程异常退出会被主进程发现并转成服务级故障。
 889: 
 890:         return (
 891:             tokenizer_manager,
```

**读这段时抓住：**

- detokenizer 子进程先启动，然后主进程初始化 TokenizerManager 或 MultiTokenizerRouter。
- `scheduler_init_result.wait_for_ready()` 是关键同步点：HTTP 层不能盲目认为模型已经可用。
- `max_req_input_len` 从 scheduler 回写给 tokenizer manager，输入长度校验才有真实模型/部署上下文。
- `SubprocessWatchdog` 把 scheduler/detokenizer 崩溃变成主进程可感知事件，是生产服务韧性的一部分。


### `_setup_and_run_http_server`：把 engine 状态挂到 FastAPI app 上

```python
# python/sglang/srt/entrypoints/http_server.py:2251-2335
2251: def _setup_and_run_http_server(
      # 注：关键调用：`_setup_and_run_http_server` - 把已经启动的 engine 组件挂到 FastAPI app 上，并启动 HTTP server。
2252:     server_args: ServerArgs,
2253:     tokenizer_manager,
2254:     template_manager,
2255:     port_args: PortArgs,
2256:     scheduler_infos: List[Dict],
2257:     subprocess_watchdog: Optional[SubprocessWatchdog],
2258:     execute_warmup_func: Callable = _execute_server_warmup,
2259:     launch_callback: Optional[Callable[[], None]] = None,
2260: ):
2261:     """Set up global state, configure middleware, and run uvicorn.
2262: 
2263:     Called by launch_server after subprocesses have been launched.
2264:     """
2265:     # Set global states
2266:     set_global_state(
      # 注：关键调用：`set_global_state` - 把 manager/template/scheduler 信息放入 HTTP 全局状态，endpoint 之后通过它拿到 runtime 对象。
2267:         _GlobalState(
      # 注：关键调用：`_GlobalState` - HTTP 全局状态对象；FastAPI endpoint 后续通过它拿 tokenizer manager、template manager 和 scheduler info。
2268:             tokenizer_manager=tokenizer_manager,
2269:             template_manager=template_manager,
2270:             scheduler_info=scheduler_infos[0],
2271:         )
2272:     )
2273: 
2274:     # Store watchdog on tokenizer_manager (single source of truth for SIGQUIT handler)
2275:     if tokenizer_manager is not None:
2276:         tokenizer_manager._subprocess_watchdog = subprocess_watchdog
2277: 
2278:     if server_args.enable_metrics:
      # 注：启用 metrics 时 HTTP 层会注入 Prometheus middleware，scheduler/cache 侧也会带指标 collector。
2279:         add_prometheus_track_response_middleware(app)
      # 注：关键调用：`add_prometheus_track_response_middleware` - 在 FastAPI 层增加 response metrics middleware，使 HTTP 请求耗时和状态码进入 Prometheus 指标。
2280: 
2281:     # Pass additional arguments to the lifespan function.
2282:     # They will be used for additional initialization setups.
2283:     if server_args.tokenizer_worker_num == 1:
      # 注：单 tokenizer worker 模式下主进程直接持有 TokenizerManager；多 worker 模式改用共享内存和 router。
2284:         # If it is single tokenizer mode, we can pass the arguments by attributes of the app object.
2285:         app.is_single_tokenizer_mode = True
      # 注：HTTP lifecycle 通过这个标志区分单 tokenizer 与多 tokenizer worker 的 warmup/路由路径。
2286:         app.server_args = server_args
      # 注：single-tokenizer 模式把 server args 挂到 FastAPI app，lifespan/warmup 线程直接读取它。
2287:         app.warmup_thread_kwargs = dict(
      # 注：HTTP server lifespan 会用这些参数执行 warmup，保证监听后模型路径已经完成必要预热。
2288:             server_args=server_args,
2289:             launch_callback=launch_callback,
2290:             execute_warmup_func=execute_warmup_func,
2291:         )
2292: 
2293:         # Add api key authorization
2294:         # This is only supported in single tokenizer mode.
2295:         #
2296:         # Backward compatibility:
2297:         # - api_key only: behavior matches legacy (all endpoints require api_key)
2298:         # - no keys: legacy had no restriction; ADMIN_FORCE endpoints must still be rejected when
2299:         #   admin_api_key is not configured.
2300:         if (
2301:             server_args.api_key
2302:             or server_args.admin_api_key
2303:             or app_has_admin_force_endpoints(app)
      # 注：关键调用：`app_has_admin_force_endpoints` - 检查当前 FastAPI app 是否暴露强制类管理 endpoint；有这类 endpoint 时必须打开 admin key 校验。
2304:         ):
2305:             from sglang.srt.utils.auth import add_api_key_middleware
2306: 
2307:             add_api_key_middleware(
      # 注：关键调用：`add_api_key_middleware` - 把 API key/admin API key 鉴权逻辑注册到 FastAPI middleware。
2308:                 app,
2309:                 api_key=server_args.api_key,
2310:                 admin_api_key=server_args.admin_api_key,
2311:             )
2312:     else:
2313:         # If it is multi-tokenizer mode, we need to write the arguments to shared memory
2314:         # for other worker processes to read.
2315:         app.is_single_tokenizer_mode = False
2316:         multi_tokenizer_args_shm = write_data_for_multi_tokenizer(
      # 注：`multi_tokenizer_args_shm` 接收 `write_data_for_multi_tokenizer` 的结果：多 tokenizer worker 模式下通过共享内存传递启动参数。
2317:             port_args, server_args, scheduler_infos[0]
2318:         )
2319: 
2320:     try:
2321:         # Update logging configs
2322:         set_uvicorn_logging_configs(server_args)
      # 注：关键调用：`set_uvicorn_logging_configs` - 把 SGLang 的日志配置同步给 uvicorn，避免 HTTP server 使用另一套默认日志格式。
2323: 
2324:         if server_args.ssl_certfile:
      # 注：提供证书时 uvicorn/granian 走 HTTPS 配置；这里仅记录 SSL 文件，实际监听仍在后续 server backend。
2325:             logger.info(
2326:                 f"SSL enabled: certfile={server_args.ssl_certfile}, "
2327:                 f"keyfile={server_args.ssl_keyfile}"
2328:             )
2329: 
2330:         # Listen for HTTP requests
2331:         if server_args.tokenizer_worker_num == 1:
2332:             if server_args.enable_http2:
      # 注：HTTP/2 需要 Granian backend；默认 uvicorn 路径只覆盖普通 HTTP/1.1 ASGI 服务。
2333:                 logger.info(
2334:                     f"Starting embedded Granian HTTP/2 server on "
2335:                     f"{server_args.host}:{server_args.port}"
```

**读这段时抓住：**

- `set_global_state` 把 tokenizer_manager/template_manager/scheduler_info 暴露给 FastAPI endpoint 依赖函数。
- single-tokenizer 模式直接把 `server_args` 和 warmup kwargs 挂到 `app`；multi-tokenizer 模式则写共享内存给 worker 读。
- API key middleware 只在 single-tokenizer 模式直接添加；多 tokenizer/多 worker 的 HTTP 启动路径不同。
- 最后根据 HTTP/2、SSL refresh、tokenizer worker 数选择 Granian 或 uvicorn 的不同启动方式。


## Manager 边界

SGLang 的默认 HTTP 服务不是一个“大函数直接 forward”。它把工作拆成几类 manager：

- `TokenizerManager`：API 请求、chat template、多模态预处理、tokenization、streaming 状态。
- `Scheduler`：waiting/running queue、prefix cache、KV pool、grammar queue、batch formation、forward 调用。
- `DetokenizerManager`：把 token ids 增量转回文本，并负责流式输出中的 offset 管理。
- `TpModelWorker` / `ModelRunner`：真正持有模型权重、attention backend、CUDA graph runner 和 KV cache pool。

这条边界非常重要：多数 Feature 要么在 tokenizer 层增加请求字段，要么在 scheduler 层改变调度/缓存，要么在 model runner/attention backend 层改变 tensor 执行。


### `TokenizerManager.__init__` 的尾部：通信、dispatcher、采样参数类

```python
# python/sglang/srt/managers/tokenizer_manager.py:545-575
 545:             start_cpu_monitor_thread("tokenizer")
      # 注：关键调用：`start_cpu_monitor_thread` - 为 tokenizer/detokenizer 进程启动 CPU 监控线程，便于定位 CPU 侧预处理或 detokenize 阻塞。
 546: 
 547:         if self.server_args.gc_warning_threshold_secs > 0.0:
      # 注：只有显式设置 GC 阈值时 tokenizer manager 才打开 GC pause 告警，避免默认日志过多。
 548:             configure_gc_warning(self.server_args.gc_warning_threshold_secs)
      # 注：关键调用：`configure_gc_warning` - 启用 GC pause 告警；tokenizer manager 是长生命周期进程，GC 卡顿会直接影响请求入站延迟。
 549:         self.soft_watchdog = Watchdog.create(
      # 注：`self.soft_watchdog` 接收 `Watchdog.create` 的结果：为 tokenizer manager 创建软 watchdog，用于发现 handle loop 或请求处理长时间无进展。
 550:             debug_name="TokenizerManager",
 551:             watchdog_timeout=self.server_args.soft_watchdog_timeout,
 552:             soft=True,
 553:             test_stuck_time=envs.SGLANG_TEST_STUCK_TOKENIZER.get(),
 554:         )
 555: 
 556:     def init_request_dispatcher(self):
 557:         self._result_dispatcher = TypeBasedDispatcher(
      # 注：TokenizerManager 的回包分发器；scheduler/detokenizer 返回的控制对象按类型进入对应 handler。
 558:             [
 559:                 (AbortReq, self._handle_abort_req),
 560:                 (OpenSessionReqOutput, self._handle_open_session_req_output),
 561:                 (
 562:                     UpdateWeightFromDiskReqOutput,
 563:                     self._handle_update_weights_from_disk_req_output,
 564:                 ),
 565:                 (FreezeGCReq, lambda x: None),
 566:                 # For handling case when scheduler skips detokenizer and forwards back to the tokenizer manager, we ignore it.
 567:                 (HealthCheckOutput, lambda x: None),
 568:                 (ActiveRanksOutput, self.update_active_ranks),
 569:             ]
 570:         )
 571:         self.init_communicators(self.server_args)
      # 注：关键调用：`self.init_communicators` - 初始化 tokenizer manager 与 scheduler/detokenizer 间的 IPC 通道。
 572: 
 573:         self.sampling_params_class = SamplingParams
      # 注：请求入站时使用的采样参数实现，后续会把用户 JSON 规范化为内部 SamplingParams。
 574:         self.signal_handler_class = SignalHandler
      # 注：TokenizerManager 使用的信号处理器类型，负责把进程信号转成服务清理动作。
 575: 
```

**读这段时抓住：**

- `TypeBasedDispatcher` 让 tokenizer manager 能处理 scheduler/detokenizer 返回的多种对象，而不是写一串 endpoint 专用回调。
- `init_communicators` 建立 ZeroMQ/IPC 通道；这就是 API 进程和 scheduler/detokenizer 进程之间的边界。
- `sampling_params_class = SamplingParams` 意味着请求参数在进入 scheduler 前已经被规范化成内部采样对象。


### `DetokenizerManager.__init__`：它不是附属函数，而是独立进程角色

```python
# python/sglang/srt/managers/detokenizer_manager.py:89-133
  89: class DetokenizerManager(MultiHttpWorkerDetokenizerMixin):
  90:     """DetokenizerManager is a process that detokenizes the token ids."""
  91: 
  92:     def __init__(
  93:         self,
  94:         server_args: ServerArgs,
  95:         port_args: PortArgs,
  96:     ):
  97:         # Init inter-process communication
  98:         self.init_ipc_channels(port_args, server_args)
  99: 
 100:         # Init tokenizer
 101:         self.init_tokenizer(server_args)
 102: 
 103:         # Init running status
 104:         self.init_running_status(server_args)
 105: 
 106:         # Init dispatcher
 107:         self.init_request_dispatcher()
 108: 
 109:     def init_ipc_channels(self, port_args: PortArgs, server_args: ServerArgs):
 110:         context = zmq.Context(2)
 111:         self.recv_from_scheduler = get_zmq_socket(
      # 注：detokenizer 的输入 IPC socket；scheduler 把 token id/batch output 发到这里。
 112:             context, zmq.PULL, port_args.detokenizer_ipc_name, True
 113:         )
 114:         # In multi-tokenizer mode, results are pushed back to each TokenizerWorker
 115:         # directly via SocketMapping inside multi_http_worker_event_loop, so the
 116:         # single send_to_tokenizer socket is unused.
 117:         if server_args.tokenizer_worker_num == 1:
      # 注：单 tokenizer worker 模式下主进程直接持有 TokenizerManager；多 worker 模式改用共享内存和 router。
 118:             self.send_to_tokenizer = get_zmq_socket(
      # 注：detokenizer 的输出 IPC socket；单 tokenizer 模式下文本增量通过它回到 TokenizerManager。
 119:                 context, zmq.PUSH, port_args.tokenizer_ipc_name, False
 120:             )
 121: 
 122:     def init_tokenizer(self, server_args: ServerArgs):
 123:         if server_args.skip_tokenizer_init:
      # 注：跳过 tokenizer 初始化时 detokenizer 不做 token-id 到文本转换，适用于 embedding/token-id-only 等路径。
 124:             self.tokenizer = None
      # 注：detokenizer 持有 tokenizer 后才能把增量 token id 解码成文本；skip 模式则保持 None。
 125:         else:
 126:             self.tokenizer = get_tokenizer(
 127:                 server_args.tokenizer_path,
 128:                 tokenizer_mode=server_args.tokenizer_mode,
 129:                 trust_remote_code=server_args.trust_remote_code,
 130:                 revision=server_args.revision,
 131:                 tokenizer_backend=server_args.tokenizer_backend,
 132:             )
 133: 
```

**读这段时抓住：**

- detokenizer 只从 scheduler 收 token id / batch output，再把文本结果发回 tokenizer manager。
- `skip_tokenizer_init` 会让它不加载 tokenizer；embedding/token-id-only 等路径可以绕开普通文本 detokenization。
- 这个边界减少 scheduler 的 CPU 文本处理负担，也让 streaming 输出可以和 GPU 调度解耦。


In [ ]:
paths = [
    "python/sglang/srt/entrypoints/engine.py",
    "python/sglang/srt/managers/tokenizer_manager.py",
    "python/sglang/srt/managers/scheduler.py",
    "python/sglang/srt/managers/detokenizer_manager.py",
]
for path in paths:
    print("\n==", path)
    for lineno, kind, name in find_defs(path):
        if name in {
            "init_tokenizer_manager", "run_scheduler_process", "run_detokenizer_process",
            "TokenizerManager", "generate_request", "Scheduler",
            "event_loop_normal", "event_loop_overlap", "run_batch", "process_batch_result",
            "DetokenizerManager",
        }:
            print(f"{lineno:4d} {kind:18s} {name}")


## 请求对象的生命周期

粗略流程：

1. FastAPI endpoint 接收 `/generate` 或 `/v1/chat/completions`。
2. OpenAI serving 层把 chat/completion 请求转为内部 `GenerateReqInput`。
3. `TokenizerManager.generate_request` 负责 tokenization、多模态输入处理、采样参数校验，并把 tokenized 请求发给 scheduler。
4. `Scheduler` 把请求放入 waiting queue；grammar 请求可能先进 `GrammarManager` 队列等编译完成。
5. `get_new_batch_prefill` 根据调度策略、KV 可用量、prefix hit、chunked prefill 等形成 `ScheduleBatch`。
6. `ForwardBatch.init_new` 将 CPU 侧调度数据变成 GPU 侧 tensor 元数据。
7. `ModelRunner.forward` 执行模型，attention 层通过 `RadixAttention` 转给当前 attention backend。
8. scheduler 处理 logits/sampling/finish reason，把结果发往 detokenizer/tokenizer manager。
9. API 层按 non-streaming 或 streaming 返回。


### `TokenizerManager.generate_request`：入站请求的主脊柱

```python
# python/sglang/srt/managers/tokenizer_manager.py:576-632
 576:     async def generate_request(
 577:         self,
 578:         obj: Union[GenerateReqInput, EmbeddingReqInput],
 579:         request: Optional[fastapi.Request] = None,
 580:     ):
 581:         self.auto_create_handle_loop()
 582: 
 583:         # Normalize the request
 584:         obj.normalize_batch_and_arguments()
      # 注：关键调用：`obj.normalize_batch_and_arguments` - 把单请求/批请求和参数别名规范化，后续路径才能统一处理。
 585:         self._set_default_priority(obj)
      # 注：关键调用：`self._set_default_priority` - 为请求补齐默认优先级，供 scheduler priority policy 使用。
 586: 
 587:         if isinstance(obj, GenerateReqInput) and obj.routed_dp_rank is not None:
 588:             dp_size = self.server_args.dp_size
      # 注：DP size 决定 routed_dp_rank 是否有效；单 DP 时 routed_dp_rank=0 会被忽略。
 589:             if dp_size <= 1 and obj.routed_dp_rank == 0:
 590:                 logger.debug(
 591:                     f"routed_dp_rank={obj.routed_dp_rank} is ignored because dp_size={dp_size}"
 592:                 )
 593:             elif obj.routed_dp_rank < 0 or obj.routed_dp_rank >= dp_size:
 594:                 raise ValueError(
 595:                     f"routed_dp_rank={obj.routed_dp_rank} out of range [0, {dp_size})"
 596:                 )
 597: 
 598:         if self.server_args.tokenizer_worker_num > 1:
      # 注：多 tokenizer worker 请求必须附带 worker IPC 信息，否则 scheduler 输出无法回到发起请求的 HTTP worker。
 599:             self._attach_multi_http_worker_info(obj)
      # 注：关键调用：`self._attach_multi_http_worker_info` - 多 tokenizer worker 模式下给请求补充 HTTP worker IPC 信息，scheduler 回包才能路由回正确 worker。
 600:         self._init_req_state(obj, request)
      # 注：关键调用：`self._init_req_state` - 创建 rid 到 ReqState 的映射，后续 streaming/non-streaming 回包都靠它找到等待者。
 601:         try:
 602:             if self.server_args.language_only:
      # 注：language-only/EPD encode 请求走 disaggregation encode 旁路，不进入普通 generate decode 流程。
 603:                 self._handle_epd_disaggregation_encode_request(obj)
      # 注：关键调用：`self._handle_epd_disaggregation_encode_request` - EPD language-only encode 请求的旁路处理；这种请求不进入普通生成式 decode 流程。
 604: 
 605:             # Log the request
 606:             self.request_logger.log_received_request(obj, self.tokenizer, request)
      # 注：关键调用：`self.request_logger.log_received_request` - 在请求刚完成规范化后记录输入摘要，便于把用户请求和后续 scheduler 日志串起来。
 607: 
 608:             async with self.is_pause_cond:
 609:                 await self.is_pause_cond.wait_for(lambda: not self.is_pause)
 610: 
 611:             async with self.model_update_lock.reader_lock:
 612:                 await self._validate_and_resolve_lora(obj)
      # 注：关键调用：`self._validate_and_resolve_lora` - 在 tokenization 前解析并校验 LoRA 选择，确保 cache key 和调度状态带上正确 LoRA 维度。
 613: 
 614:                 # Tokenize the request and send it to the scheduler
 615:                 if obj.is_single:
      # 注：单请求直接 tokenize/send/wait；批请求要拆分内部 request state 并复用 batch handler。
 616:                     tokenized_obj = await self._tokenize_one_request(obj)
      # 注：单请求完成 tokenizer/chat template/multimodal 处理后的内部请求对象，下一步会通过 IPC 发送给 scheduler。
 617:                     state = self.rid_to_state[obj.rid]
      # 注：`rid_to_state` 中的请求等待状态；找到它才能把 scheduler 输出送回对应 API coroutine。
 618:                     if obj.return_prompt_token_ids:
      # 注：用户要求返回 prompt token id 时，tokenization 结果需要暂存在 ReqState 里供最终响应组装。
 619:                         state.prompt_token_ids = list(tokenized_obj.input_ids)
      # 注：用户要求回传 prompt token ids 时，把 tokenization 结果挂到 ReqState，最终响应组装会读取它。
 620:                     self._send_one_request(tokenized_obj)
      # 注：关键调用：`self._send_one_request` - 把 tokenized request 通过 IPC 发给 scheduler。
 621:                     async for response in self._wait_one_response(obj, request):
      # 注：关键调用：`self._wait_one_response` - 等待 scheduler/detokenizer 回包，并按 streaming/non-streaming 产出 API 响应。
 622:                         yield response
 623:                 else:
 624:                     async for response in self._handle_batch_request(obj, request):
 625:                         yield response
 626:         except Exception:
 627:             # _init_req_state created a rid_to_state entry per (sub-)request up
 628:             # front. The normal remover is the scheduler-response path
 629:             # (_handle_batch_output), so a failure *before* a request reaches the
 630:             # scheduler -- e.g. input-length validation rejecting an over-context
 631:             # request -- would otherwise leak those entries forever. Drop any that
 632:             # are still pending; entries already removed on the normal completion
```

**读这段时抓住：**

- `auto_create_handle_loop()` 确保后台回包循环已经启动，否则请求发出去后没人消费 scheduler/detokenizer 输出。
- `normalize_batch_and_arguments()` 把单请求/批请求、多种参数别名规范化，这是后续代码能统一处理的前提。
- `model_update_lock.reader_lock` 保护权重/LoRA 更新期间的请求一致性；请求路径并不只处理文本。
- 单请求路径是 tokenize -> `_send_one_request` -> `_wait_one_response`；批请求走 `_handle_batch_request`，但仍会拆成内部 request state。
- 异常分支会清理 `rid_to_state`，这是防止早期校验失败导致内存状态泄漏的关键。


### `TokenizerManager.handle_loop`：为什么响应能回到原请求

```python
# python/sglang/srt/managers/tokenizer_manager.py:1824-1888
1824:     async def handle_loop(self):
1825:         """The event loop that handles requests"""
1826:         while True:
1827:             with self.soft_watchdog.disable():
1828:                 recv_obj = await self.recv_from_detokenizer.recv_pyobj()
      # 注：TokenizerManager 从 detokenizer 收到的 batch 输出或控制对象，是 API streaming/non-streaming 回包的来源。
1829:             if isinstance(
1830:                 recv_obj,
1831:                 (BatchStrOutput, BatchEmbeddingOutput, BatchTokenIDOutput),
1832:             ):
1833:                 await self._handle_batch_output(recv_obj)
      # 注：关键调用：`self._handle_batch_output` - 把 batch 输出拆回每个 rid，并整理 meta_info、文本、logprob、finish reason。
1834:             else:
1835:                 self._result_dispatcher(recv_obj)
      # 注：关键调用：`self._result_dispatcher` - 处理非 batch-output 的控制消息，例如 abort、flush cache、profile 等 manager 间响应。
1836:             self.last_receive_tstamp = real_time()
1837:             self.soft_watchdog.feed()
1838: 
1839:     async def _handle_batch_output(
1840:         self,
1841:         recv_obj: Union[
1842:             BatchStrOutput,
1843:             BatchEmbeddingOutput,
1844:             BatchTokenIDOutput,
1845:         ],
1846:     ):
1847:         pending_notify: dict[str, ReqState] = {}
      # 注：批量通知暂存表，用于减少逐请求唤醒造成的 event-loop 抖动。
1848:         batch_notify_size = self.server_args.batch_notify_size
      # 注：控制一次合并通知多少请求，影响 streaming 高并发下的唤醒粒度。
1849:         for i, rid in enumerate(recv_obj.rids):
1850:             state = self.rid_to_state.get(rid, None)
      # 注：`rid_to_state` 中的请求等待状态；找到它才能把 scheduler 输出送回对应 API coroutine。
1851:             if state is None:
      # 注：收到的 rid 已不在 `rid_to_state`，通常是请求已 abort/health check 已提前清理，需要避免把孤儿输出写回用户流。
1852:                 # Known race: /health_generate pops its rid as soon as ANY message bumps last_receive_tstamp.
1853:                 if rid.startswith(HEALTH_CHECK_RID_PREFIX):
      # 注：health check 请求可能只等任意回包更新时间戳，rid 已清理时静默跳过是预期竞争。
1854:                     continue
1855:                 logger.error(
1856:                     f"Received output for {rid=} but the state was deleted in TokenizerManager."
1857:                 )
1858:                 continue
1859: 
1860:             # Build meta_info and return value
1861:             meta_info = {
      # 注：用户响应中的元信息在 tokenizer manager 组装，包含 finish reason、prompt token 数和可选 timing/logprob。
1862:                 "id": rid,
1863:                 "finish_reason": recv_obj.finished_reasons[i],
1864:                 "prompt_tokens": recv_obj.prompt_tokens[i],
1865:                 "weight_version": self.server_args.weight_version,
1866:                 "num_retractions": recv_obj.retraction_counts[i],
1867:             }
1868: 
1869:             if self.enable_metrics:
      # 注：开启 metrics 时把 scheduler 侧 time_stats 合并进 API meta_info，用户响应才带性能分解。
1870:                 if recv_obj.time_stats is not None:
      # 注：只有 scheduler 返回了 per-request timing 时，tokenizer manager 才能把它转换成输出 meta_info。
1871:                     scheduler_time_stats = recv_obj.time_stats[i]
      # 注：scheduler 返回的 per-request 执行耗时，稍后会转换为输出 meta_info。
1872:                     meta_info.update(scheduler_time_stats.convert_to_output_meta_info())
1873: 
1874:             if getattr(state.obj, "return_logprob", False):
      # 注：用户请求 logprob 时，tokenizer manager 需要把 token id/logprob 转成 OpenAI/SGLang API 风格。
1875:                 self.convert_logprob_style(
      # 注：关键调用：`self.convert_logprob_style` - 把 scheduler 返回的 logprob token 信息转换成用户 API 请求的文本/token 风格。
1876:                     meta_info,
1877:                     state,
1878:                     state.obj.top_logprobs_num,
1879:                     state.obj.token_ids_logprob,
1880:                     state.obj.return_text_in_logprobs
1881:                     and not self.server_args.skip_tokenizer_init,
1882:                     recv_obj,
1883:                     i,
1884:                 )
1885: 
1886:             if not isinstance(recv_obj, BatchEmbeddingOutput):
      # 注：embedding 输出没有生成文本 finish 语义，文本生成相关 meta_info 只加到非 embedding 响应。
1887:                 meta_info.update(
1888:                     {
```

**读这段时抓住：**

- 它一直从 detokenizer/socket 读对象；Batch output 走 `_handle_batch_output`，控制类对象交给 dispatcher。
- `rid_to_state` 是回包路由表：scheduler 只知道 request id，API 层等待的是对应 state 的 event/output list。
- `meta_info` 在这里组装，所以许多用户可见统计信息并不是 model runner 直接返回的。
- streaming 的增量 offset、logprob 文本化、finish reason 都在这一层被整理成 API 输出。


In [ ]:
for path, names in [
    ("python/sglang/srt/managers/tokenizer_manager.py", {"generate_request", "_tokenize_one_request"}),
    ("python/sglang/srt/managers/scheduler.py", {"get_new_batch_prefill", "run_batch", "process_batch_result"}),
    ("python/sglang/srt/managers/schedule_batch.py", {"ScheduleBatch", "ForwardMode", "Req"}),
]:
    print("\n==", path)
    for row in find_defs(path, names=names):
        print(row)


### `engine.py` 初始化边界：manager 进程在这里被拼起来

```python
# python/sglang/srt/entrypoints/engine.py:117-175
 117: class SchedulerInitResult:
 118:     """Result from launching schedulers."""
 119: 
 120:     scheduler_infos: List[Dict[str, Any]]
 121:     all_child_pids: List[int] = dataclasses.field(default_factory=list)
 122:     wait_for_ready: Callable[[], None] = lambda: None
 123:     wait_for_completion: Callable[[], None] = lambda: None
 124:     engine_info_bootstrap_server: Optional[Any] = None
 125: 
 126: 
 127: def init_tokenizer_manager(
 128:     server_args: ServerArgs,
 129:     port_args: PortArgs,
 130:     TokenizerManagerClass: Optional[TokenizerManager] = None,
 131: ) -> Tuple[TokenizerManager, TemplateManager]:
 132:     # Launch tokenizer process
 133:     TokenizerManagerClass = TokenizerManagerClass or TokenizerManager
      # 注：测试或私有 fork 可以替换 TokenizerManager 实现；默认使用标准 TokenizerManager。
 134:     tokenizer_manager = TokenizerManagerClass(server_args, port_args)
      # 注：主进程中的 TokenizerManager，负责 API 入站、tokenization 和等待 scheduler/detokenizer 回包。
 135: 
 136:     # Initialize templates
 137:     template_manager = TemplateManager()
      # 注：chat/completion 模板管理器，初始化后还会给 auto parser 提供推断结果。
 138:     template_manager.initialize_templates(
      # 注：关键调用：`template_manager.initialize_templates` - 根据 tokenizer 和 server args 初始化 chat/completion template，并顺带推断 reasoning/tool parser。
 139:         tokenizer_manager=tokenizer_manager,
 140:         model_path=server_args.model_path,
 141:         chat_template=server_args.chat_template,
 142:         completion_template=server_args.completion_template,
 143:     )
 144: 
 145:     # Resolve any remaining auto parsers using template manager's detection results
 146:     for attr, suggested, label in (
 147:         (
 148:             "reasoning_parser",
 149:             template_manager.suggested_reasoning_parser,
 150:             "reasoning parser",
 151:         ),
 152:         (
 153:             "tool_call_parser",
 154:             template_manager.suggested_tool_call_parser,
 155:             "tool-call parser",
 156:         ),
 157:     ):
 158:         if getattr(server_args, attr) != "auto":
      # 注：用户显式指定 parser 时不覆盖；只有 auto 才使用 template manager 的推断结果。
 159:             continue
 160:         if suggested is not None:
      # 注：template manager 成功从 chat template 推断 parser，就把 auto 改写成具体 parser 名称。
 161:             setattr(server_args, attr, suggested)
 162:             logger.info(
 163:                 f"Auto-detected --{attr.replace('_', '-')} as '{suggested}' from chat template"
 164:             )
 165:         else:
 166:             logger.warning(
 167:                 f"--{attr.replace('_', '-')}=auto specified but could not detect "
 168:                 f"{label} from chat template. Disabling {label}."
 169:             )
 170:             setattr(server_args, attr, None)
 171: 
 172:     return tokenizer_manager, template_manager
 173: 
 174: 
 175: class Engine(EngineScoreMixin, EngineBase):
```

**读这段时抓住：**

- `SchedulerInitResult` 是 scheduler 初始化后回传给 HTTP/tokenizer 侧的能力摘要，例如 max input length、model info。
- `init_tokenizer_manager` 不是单纯构造 tokenizer，它会启动 scheduler/detokenizer 进程并建立 IPC 名称。
- 如果一个 Feature 需要跨 manager 传递新字段，通常要顺着这里确认哪个进程先知道它。


## 读 `Scheduler` 时不要迷路

`scheduler.py` 很长，因为它同时处理普通 decode、overlap schedule、grammar、speculative decoding、disaggregation、LoRA、profiling、metrics、weight update 等路径。建议按这几个函数读：

- `__init__`：所有子系统如何挂到 scheduler 上。
- `event_loop_normal` / `event_loop_overlap`：主循环形状。
- `get_new_batch_prefill`：prefill batch 进入 GPU 的入口。
- `run_batch`：从 `ScheduleBatch` 到 model worker forward。
- `process_batch_result`：采样、finish、streaming、cache 更新。


### `Scheduler.event_loop_normal`：普通调度主循环的形状

```python
# python/sglang/srt/managers/scheduler.py:1506-1535
1506:     def event_loop_normal(self):
1507:         """A normal scheduler loop."""
1508:         while True:
1509:             if self.gracefully_exit:
      # 注：scheduler 收到优雅退出信号后跳出主循环，不再接收新 batch。
1510:                 break
1511: 
1512:             # Receive requests
1513:             recv_reqs = self.request_receiver.recv_requests()
      # 注：scheduler 本轮从 tokenizer manager 收到的新请求和控制消息。
1514:             self.process_input_requests(recv_reqs)
      # 注：关键调用：`self.process_input_requests` - 把新请求、abort、flush、update weight 等控制消息并入 scheduler 内部队列。
1515:             if self._engine_paused:
      # 注：engine pause 时只接收控制消息，不继续组 batch 或调用 model worker。
1516:                 continue
1517: 
1518:             # Get the next batch to run
1519:             batch = self.get_next_batch_to_run()
      # 注：scheduler 本轮选择出的可执行 batch，可能来自 running decode 或 waiting prefill。
1520:             self.cur_batch = batch
      # 注：scheduler 当前正在执行/准备执行的 batch；abort、pause、metrics 等逻辑会读取它。
1521: 
1522:             # Launch the current batch
1523:             if batch:
      # 注：存在可运行 batch 时进入 GPU forward；没有 batch 时进入 idle 维护路径。
1524:                 result = self.run_batch(batch)
      # 注：model worker forward/sampling 的输出，下一步必须交给 `process_batch_result` 更新请求和 cache 状态。
1525:                 self.process_batch_result(batch, result)
      # 注：关键调用：`self.process_batch_result` - 把 forward/sampling 结果写回请求状态、发送输出并维护 cache。
1526:             else:
1527:                 # When the server is idle, do self-check and re-init some states.
1528:                 self.on_idle()
      # 注：关键调用：`self.on_idle` - scheduler 无 batch 可跑时执行空闲维护，例如健康检查、状态重置或等待新请求。
1529: 
1530:             # Update last_batch
1531:             self.last_batch = batch
      # 注：记录上一轮 batch，overlap/metrics/调试路径会用它理解上一轮执行状态。
1532:             if envs.SGLANG_ENABLE_STRICT_MEM_CHECK_DURING_BUSY.get():
      # 注：调试内存泄漏时在 busy loop 中强制检查 KV pool/radix cache 不变量，生产默认关闭。
1533:                 self.invariant_checker.self_check_during_busy()
1534: 
1535:     @DynamicGradMode()
```

**读这段时抓住：**

- 循环每轮先 `recv_requests`，再处理 grammar-ready、running batch、new prefill batch 等状态。
- `forward_ct` 和 watchdog 让 scheduler 可以暴露健康状态；调度循环本身也是生产系统对象。
- 普通循环和 overlap 循环分开，是因为 overlap schedule 要把 CPU 调度和 GPU forward 的依赖拆得更细。


### `Scheduler.event_loop_overlap`：重叠调度不是简单开线程

```python
# python/sglang/srt/managers/scheduler.py:1536-1565
1536:     def event_loop_overlap(self):
1537:         """A scheduler loop that overlaps the CPU processing and GPU computation."""
1538:         self.result_queue: Deque[
      # 注：overlap 调度暂存上一轮 GPU 结果，使 CPU 可以先准备下一轮 batch 再回收上一轮输出。
1539:             Tuple[ScheduleBatch, Union[GenerationBatchResult, EmbeddingBatchResult]]
1540:         ] = deque()
1541: 
1542:         def pop_and_process():
1543:             # Process the results of the last batch
1544:             tmp_batch, tmp_result = self.result_queue.popleft()
      # 注：overlap 路径中出队的上一轮结果，必须先 process 后才能安全推进相关请求状态。
1545:             self.process_batch_result(tmp_batch, tmp_result)
      # 注：关键调用：`self.process_batch_result` - 把 forward/sampling 结果写回请求状态、发送输出并维护 cache。
1546: 
1547:         while True:
1548:             if self.gracefully_exit:
      # 注：scheduler 收到优雅退出信号后跳出主循环，不再接收新 batch。
1549:                 break
1550: 
1551:             # Receive requests
1552:             recv_reqs = self.request_receiver.recv_requests()
      # 注：scheduler 本轮从 tokenizer manager 收到的新请求和控制消息。
1553:             self.process_input_requests(recv_reqs)
      # 注：关键调用：`self.process_input_requests` - 把新请求、abort、flush、update weight 等控制消息并入 scheduler 内部队列。
1554:             if self._engine_paused:
      # 注：engine pause 时只接收控制消息，不继续组 batch 或调用 model worker。
1555:                 continue
1556: 
1557:             self._apply_war_barrier()
      # 注：关键调用：`self._apply_war_barrier` - overlap 调度里处理 write-after-read 依赖屏障，避免上一轮 GPU 结果尚未安全发布就复用资源。
1558: 
1559:             # Get the next batch to run
1560:             batch = self.get_next_batch_to_run()
      # 注：scheduler 本轮选择出的可执行 batch，可能来自 running decode 或 waiting prefill。
1561:             self.cur_batch = batch
      # 注：scheduler 当前正在执行/准备执行的 batch；abort、pause、metrics 等逻辑会读取它。
1562:             disable_overlap_for_batch = self.is_disable_overlap_for_batch(batch)
      # 注：某些 batch 因依赖或特殊 feature 不能 overlap，这个标志决定是否立即处理上一轮结果。
1563: 
1564:             # If we do not need to overlap the current batch with the last batch,
1565:             # we can process the last batch immediately.
```

**读这段时抓住：**

- overlap 路径有专门的 `result_queue`，上一轮 GPU 结果可能在下一轮 CPU 调度时才回收。
- 许多 Feature 要分别验证 normal/overlap 两条路径，例如 speculative、grammar、abort、pause generation。
- 读 scheduler bug 时，要先确认当前服务是否启用了 overlap，否则你可能在读错主循环。


In [ ]:
show_lines("python/sglang/srt/managers/scheduler.py", 1500, 1565)


## 小练习：定位一次 `/generate` 的关键函数

下面的 cell 不启动服务，只用 AST 找出关键定义。读源码时可以把这些行号作为入口点。


In [ ]:
targets = {
    "python/sglang/srt/entrypoints/http_server.py": {"generate_request", "launch_server"},
    "python/sglang/srt/managers/tokenizer_manager.py": {"generate_request"},
    "python/sglang/srt/managers/scheduler.py": {"get_new_batch_prefill", "run_batch", "process_batch_result"},
}
for path, names in targets.items():
    print("\n" + path)
    for lineno, kind, name in find_defs(path, names):
        print(f"  {name}: line {lineno}")
